# Nemotron Submission Packaging — 2026-04-28

Purpose:
- keep the known-good submission packaging flow frozen
- produce a valid `submission.zip` for Kaggle
- separate submission generation from local prompt evaluation

Current project conventions:
- `P3_metric_aligned` is the best local prompt template from offline eval
- however, this notebook is submission-only and does not run prompt evaluation
- this notebook creates a compatible LoRA adapter shell for Nemotron-3-Nano-30B


In [ ]:
import os
import json
import shutil
import zipfile
from pathlib import Path

import kagglehub
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

OUTPUT_DIR = '/kaggle/working'
ADAPTER_DIR = os.path.join(OUTPUT_DIR, 'nemotron_lora_adapter')
ZIP_PATH = os.path.join(OUTPUT_DIR, 'submission.zip')
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')

if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print('MODEL_PATH:', MODEL_PATH)
print('ADAPTER_DIR:', ADAPTER_DIR)

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch_dtype,
    trust_remote_code=True,
    device_map='auto',
)

# Keep this aligned with competition constraints.
# Rank must be <= 32.
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)
model.save_pretrained(ADAPTER_DIR)

print('Saved files:', os.listdir(ADAPTER_DIR))
required = {'adapter_config.json', 'adapter_model.safetensors'}
present = set(os.listdir(ADAPTER_DIR))
missing = required - present
if missing:
    raise ValueError(f'Missing required adapter files: {missing}')

print('Adapter shell ready.')


In [ ]:
required_files = ['adapter_config.json', 'adapter_model.safetensors']

for fname in required_files:
    full_path = os.path.join(ADAPTER_DIR, fname)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f'Required file missing: {full_path}')

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        full_path = os.path.join(ADAPTER_DIR, fname)
        zf.write(full_path, arcname=fname)

print('Created:', ZIP_PATH)
print('Zip size (MB):', os.path.getsize(ZIP_PATH) / 1024 / 1024)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    print('Zip contents:', zf.namelist())

assert os.path.exists(ZIP_PATH), 'submission.zip was not created'
print('Done. This notebook is submission-only.')
